In [1]:
import pandas as pd

df = pd.read_csv('social_media_user_behavior.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (2000, 34)


,user_id,age,gender,country,profession,primary_platform,platforms_used_count,daily_usage_hours,sessions_per_day,avg_session_duration_min,...,primary_purpose,sleep_disruption,self_reported_mental_health_score,screen_time_concern,notification_frequency,privacy_setting,influencer_status,account_join_date,mood_while_scrolling,takes_social_media_breaks
0,USR00001,30,Female,UK,Doctor,Facebook,1,1.6,2,48.0,...,Entertainment,No impact,3.9,No,Do Not Disturb,Friends Only,No,2022-10-21,Neutral,No
1,USR00002,25,Male,USA,Entrepreneur,Instagram,2,2.4,5,28.8,...,News & Updates,Moderate impact,4.2,No,Selected,Public,No,2024-12-03,Neutral,Yes
2,USR00003,32,Male,UAE,Freelancer,TikTok,5,0.5,6,5.0,...,Learning,Mild impact,4.8,Yes,Always On,Friends Only,No,2023-03-29,Happy,Yes
3,USR00004,39,Prefer not to say,Pakistan,Other,TikTok,4,3.0,7,25.7,...,Entertainment,Mild impact,7.9,Somewhat,Always On,Public,Yes,2024-04-26,Happy,No
4,USR00005,25,Male,UK,Other,YouTube,3,3.9,7,33.4,...,Entertainment,Mild impact,6.5,No,Do Not Disturb,Public,No,2023-09-08,Inspired,No


## Dataset Summary

This dataset comes from Kaggle and was synthetically generated to simulate realistic social media user behavior patterns across a global population. It was designed to mirror the kinds of behavioral and psychological data that social media researchers and platforms collect through surveys, app telemetry, and user profiling. While synthetic, the distributions are modeled on real-world trends in digital behavior research. Social media is one of the defining features of modern life, yet we rarely get a comprehensive picture of how usage patterns connect to wellbeing. This dataset is uniquely interesting because it bridges behavioral data with psychological self-reports. This makes it possible to explore questions that matter deeply in public health, marketing, and technology design. Such as whether or not heavy users feel worse about their screen time or taking breaks actually correlates with better mental health.

---
## Question 1: Which social media platform is associated with the highest average daily usage?

In [2]:
avg_usage_by_platform = (
    df.groupby('primary_platform')['daily_usage_hours']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
print(avg_usage_by_platform)

primary_platform
LinkedIn     3.14
YouTube      3.10
Instagram    3.09
Facebook     3.05
Snapchat     2.99
Pinterest    2.91
TikTok       2.89
Twitter/X    2.82
Name: daily_usage_hours, dtype: float64


Users whose primary platform is LinkedIn spend the most time on social media on average, followed closely by YouTube and Instagram. Interestingly, TikTok actually ranks near the bottom in this dataset. The differences between platforms are relatively minute, suggesting that daily usage is more driven by individual habits rather than platform design.

---
## Question 2: Do users who report sleep disruption from social media have lower mental health scores?

In [3]:
mental_health_by_sleep = (
    df.groupby('sleep_disruption')['self_reported_mental_health_score']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
print(mental_health_by_sleep)

sleep_disruption
Moderate impact    6.25
No impact          6.20
Severe impact      6.14
Mild impact        6.10
Name: self_reported_mental_health_score, dtype: float64


Surprisingly, the differences in mental health scores across sleep disruption levels are minimal. Users with moderate impact on sleep actually report slightly higher mental health scores than those with severe impact, which does follow the expected direction. However, users with no sleep impact fall in the middle. This suggests that in this dataset, sleep disruption from social media does not strongly predict worse self-reported mental health on its own.

---
## Question 3: How do influencers differ from regular users in followers, posting frequency, and daily usage?

In [4]:
influencer_comparison = (
    df.groupby('influencer_status')[['followers_count', 'posts_per_week', 'daily_usage_hours']]
    .mean()
    .round(2)
)
print(influencer_comparison)

                   followers_count  posts_per_week  daily_usage_hours
influencer_status                                                    
No                         1383.24            2.96               2.98
Yes                       40961.82            2.99               3.15


The most dramatic difference is in followers. Influencers average ~40,962 followers versus ~1,383 for non-influencers, nearly a 30× difference. However, influencers post only marginally more per week and spend only slightly more time on social media daily. This suggests that in this dataset, influencer status is primarily defined by audience size rather than posting volume or time spent.

---
## Question 4: What is the most common mood users report while scrolling?

In [5]:
mood_counts = df['mood_while_scrolling'].value_counts()
mood_pct = (mood_counts / len(df) * 100).round(1)

mood_summary = pd.DataFrame({'Count': mood_counts, 'Percentage': mood_pct.astype(str) + '%'})
print(mood_summary)

                      Count Percentage
mood_while_scrolling                  
Happy                   351      17.5%
Bored                   348      17.4%
Inspired                340      17.0%
Relaxed                 333      16.7%
Neutral                 322      16.1%
Stressed                306      15.3%


The distribution of moods is relatively even. Each of the six moods account for roughly 15–18% of users. Happy edges out first, followed closely by Bored and Inspired. The near-equal split is notable; it undermines the narrative that social media is uniformly positive or harmful. Users experience the full emotional spectrum while scrolling, and no single mood takes precedence.

---
## Question 5: Do older users worry more about their screen time than younger users?

In [6]:
# Create age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[12, 17, 24, 34, 57],
    labels=['Teen (13-17)', 'Young Adult (18-24)', 'Adult (25-34)', 'Older Adult (35+)']
)

# Crosstab with row-wise normalization (percentage within each age group)
screen_time_crosstab = pd.crosstab(
    df['age_group'],
    df['screen_time_concern'],
    normalize='index'
)

# Format as percentages
screen_time_crosstab = (screen_time_crosstab * 100).round(1)
screen_time_crosstab.columns.name = 'Screen Time Concern'
print(screen_time_crosstab)

Screen Time Concern    No  Somewhat   Yes
age_group                                
Teen (13-17)         35.7      23.9  40.4
Young Adult (18-24)  37.2      24.1  38.7
Adult (25-34)        35.6      23.0  41.3
Older Adult (35+)    32.0      25.4  42.6


The crosstab shows a gradual increase in screen time concerns relative with age. Among Teens, about 40.4% say "Yes" to screen time concern. This rises to 42.6% among Older Adults. Conversely, older adults are less likely to say "No" to concern and while the differences are not dramatic; the trend consistently shows that older users are somewhat more likely to worry about their screen time.